# 异步数据库访问

学习目标：使用 SQLAlchemy 的异步接口读写 SQLite，对照同步接口的结果，验证事务回滚、并发任务的会话归属和资源关闭。

前置知识：SQL、SQLAlchemy 会话与事务、FastAPI 路由和依赖注入、协程与异步上下文管理器。

环境准备：[FastAPI 环境与运行说明](README.md)。

运行环境：Python 3.12、SQLAlchemy 2.0、aiosqlite 0.22.1 与 FastAPI；模型采用 SQLAlchemy 2.x 的 Mapped 写法。

工作目录：content/Web与应用开发/FastAPI。全部代码直接在 Notebook 重启内核后从上到下执行，不需要启动网络服务。每次应用生命周期都创建独立的临时 SQLite 文件，退出时先关闭会话与连接池，再删除临时目录。

## 1 从一张表和一次异步写入开始

只定义一张 note 表。id 是主键，title 的 unique=True 创建数据库唯一约束，后面用重复标题触发真实写入失败。Mapped 表达映射属性的类型，mapped_column 配置数据库列。

In [1]:
from sqlalchemy import select
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column


class Base(DeclarativeBase):
    pass


class Note(Base):
    __tablename__ = "note"
    id: Mapped[int] = mapped_column(primary_key=True)
    title: Mapped[str] = mapped_column(unique=True)


print(list(Note.__table__.columns.keys()))  # 预期：['id', 'title']。

['id', 'title']


三个对象各有职责。创建工厂可以复用配置，每次调用工厂都得到新的会话。

| 名称 | 中文名称／含义 | 本章用途 |
| --- | --- | --- |
| AsyncEngine | 异步引擎 | 管理数据库连接和连接池 |
| async_sessionmaker | 异步会话工厂 | 使用固定配置创建 AsyncSession |
| AsyncSession | 异步会话 | 跟踪对象修改并执行查询、提交与回滚 |

sqlite+aiosqlite 指定 SQLite 数据库和 aiosqlite 驱动；URL.create 用 database 参数传入文件路径。aiosqlite 每个连接使用一个后台线程执行 SQLite 操作，提供 await 接口；SQLite 本身仍是本地文件数据库，不因此变成网络数据库。

In [2]:
from pathlib import Path
from tempfile import TemporaryDirectory

from sqlalchemy import URL
from sqlalchemy.ext.asyncio import (
    AsyncSession, async_sessionmaker, create_async_engine,
)

先建表，再写入一条记录。run_sync 在异步连接中调用建表 API；工厂的 begin() 同时提供新会话和事务，正常退出时提交并关闭会话。add() 登记待保存对象，不需要 await。

expire_on_commit=False 让已加载属性在提交后保持可读。结束时显式 await engine.dispose() 关闭池中连接，再由 TemporaryDirectory 删除文件。

In [3]:
with TemporaryDirectory(prefix="fastapi21-one-") as folder:
    path = Path(folder) / "notes.sqlite3"
    engine = create_async_engine(URL.create("sqlite+aiosqlite", database=str(path)))
    try:
        async with engine.begin() as connection:
            await connection.run_sync(Base.metadata.create_all)
        sessions = async_sessionmaker(engine, expire_on_commit=False)
        async with sessions.begin() as session:
            note = Note(title="开始学习")
            session.add(note)
        print({"id": note.id, "title": note.title})  # 预期：{'id': 1, 'title': '开始学习'}。
    finally:
        await engine.dispose()
assert not Path(folder).exists()
print("临时数据库已清理")  # 预期：关闭引擎并退出临时目录后显示清理提示。
# 提交后的已加载属性可读，资源按会话、连接池、目录的顺序关闭。

{'id': 1, 'title': '开始学习'}
临时数据库已清理


## 2 为异步接口管理引擎和请求会话

把同样的准备与清理放入 lifespan。应用复用引擎和会话工厂；一次请求取得一个新 AsyncSession。临时文件用于本章对照实验，每次重新进入生命周期都从空表开始。

In [4]:
from contextlib import asynccontextmanager
from typing import Annotated

from fastapi import Depends, FastAPI, HTTPException, Request
from sqlalchemy.exc import IntegrityError


Titles = list[str]
# 输入为标题字符串组成的 JSON 数组；本章只比较会话与事务行为。

建表完成后再允许处理请求；即使启动或请求过程中发生异常，finally 仍负责释放引擎。本例将文件位置放在应用状态中，便于观察退出后的清理结果。

In [5]:
@asynccontextmanager
async def async_lifespan(app: FastAPI):
    with TemporaryDirectory(prefix="fastapi21-async-") as folder:
        path = Path(folder) / "notes.sqlite3"
        engine = create_async_engine(URL.create("sqlite+aiosqlite", database=str(path)))
        app.state.database_path = path
        try:
            async with engine.begin() as connection:
                await connection.run_sync(Base.metadata.create_all)
            app.state.sessions = async_sessionmaker(engine, expire_on_commit=False)
            yield
        finally:
            await engine.dispose()


async_app = FastAPI(lifespan=async_lifespan)

yield 依赖为每次请求打开并关闭会话。工厂可以由请求共同使用，同一个 AsyncSession 不能被多个并发任务共同操作。

In [6]:
async def get_async_session(request: Request):
    async with request.app.state.sessions() as session:
        yield session


AsyncDB = Annotated[AsyncSession, Depends(get_async_session)]

POST /notes 把这一批标题放入同一个事务：全部成功才提交，任一写入失败就回滚。唯一约束失败在事务上下文退出后转换成 409，响应只给出业务说明。

In [7]:
@async_app.post("/notes", status_code=201)
async def create_async_notes(titles: Titles, session: AsyncDB):
    notes = [Note(title=title) for title in titles]
    try:
        async with session.begin():
            session.add_all(notes)
    except IntegrityError:
        raise HTTPException(status_code=409, detail="标题已存在") from None
    return [{"id": note.id, "title": note.title} for note in notes]

GET /notes 显式等待查询完成，并按 id 排序。await session.scalars() 返回本例查询的缓冲结果，随后 all() 是普通调用；读取已加载的 id、title 不再发起查询。

In [8]:
@async_app.get("/notes")
async def read_async_notes(session: AsyncDB):
    result = await session.scalars(select(Note).order_by(Note.id))
    return [{"id": note.id, "title": note.title} for note in result.all()]

ASGITransport 直接调用应用，不打开真实端口，也不会自动触发 lifespan。LifespanManager 负责启动与关闭，将 manager.app 交给客户端；这里用 asyncio.timeout 将调用等待限制在 5 秒内。

In [9]:
import asyncio

from asgi_lifespan import LifespanManager
import httpx


async with LifespanManager(async_app) as manager:
    transport = httpx.ASGITransport(app=manager.app)
    async with httpx.AsyncClient(transport=transport, base_url="http://test") as client:
        async with asyncio.timeout(5):
            created = await client.post("/notes", json=["理解事务", "关闭资源"])
            listed = await client.get("/notes")
assert created.status_code == 201 and listed.status_code == 200
async_rows = listed.json()
assert async_rows == created.json()
print(async_rows)  # 预期：[{'id': 1, 'title': '理解事务'}, {'id': 2, 'title': '关闭资源'}]。
assert not async_app.state.database_path.parent.exists()

[{'id': 1, 'title': '理解事务'}, {'id': 2, 'title': '关闭资源'}]


## 3 用独立的同步接口对照结果

同步版本使用自己的 SQLite 文件、Engine 和 Session 工厂。FastAPI 的普通 def 路由由线程池执行；会话在路由调用内部创建和关闭，不在并发请求之间共享。

文件数据库连接设置 check_same_thread=False，允许 SQLite 连接按框架调用方式使用；这不赋予 Session 并发共享能力。

In [10]:
from fastapi.testclient import TestClient
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker


@asynccontextmanager
async def sync_lifespan(app: FastAPI):
    with TemporaryDirectory(prefix="fastapi21-sync-") as folder:
        path = Path(folder) / "notes.sqlite3"
        engine = create_engine(
            URL.create("sqlite", database=str(path)),
            connect_args={"check_same_thread": False},
        )
        app.state.database_path = path
        try:
            Base.metadata.create_all(engine)
            app.state.sessions = sessionmaker(engine, expire_on_commit=False)
            yield
        finally:
            engine.dispose()

C:\Users\ZHUANG\miniconda3\envs\hands-on-computing\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


POST 保持相同输入与事务边界。会话工厂的 begin() 正常提交、异常回滚，并在退出时关闭本次会话。

In [11]:
sync_app = FastAPI(lifespan=sync_lifespan)


@sync_app.post("/notes", status_code=201)
def create_sync_notes(titles: Titles, request: Request):
    notes = [Note(title=title) for title in titles]
    try:
        with request.app.state.sessions.begin() as session:
            session.add_all(notes)
    except IntegrityError:
        raise HTTPException(status_code=409, detail="标题已存在") from None
    return [{"id": note.id, "title": note.title} for note in notes]

同步 GET 在自己的会话范围内完成查询与响应数据整理。

In [12]:
@sync_app.get("/notes")
def read_sync_notes(request: Request):
    with request.app.state.sessions() as session:
        notes = session.scalars(select(Note).order_by(Note.id)).all()
        return [{"id": note.id, "title": note.title} for note in notes]

给两个独立数据库相同输入，对照状态码和查询结果。这个小实验核对行为，不据此推断哪种实现更快。

In [13]:
with TestClient(sync_app) as client:
    created = client.post("/notes", json=["理解事务", "关闭资源"])
    listed = client.get("/notes")
assert created.status_code == 201 and listed.status_code == 200
sync_rows = listed.json()
assert sync_rows == created.json() == async_rows
print("同步与异步结果一致：", sync_rows)  # 预期：同步与异步结果一致： [{'id': 1, 'title': '理解事务'}, {'id': 2, 'title': '关闭资源'}]。
assert not sync_app.state.database_path.parent.exists()

同步与异步结果一致： [{'id': 1, 'title': '理解事务'}, {'id': 2, 'title': '关闭资源'}]


## 4 写入失败后验证整个事务回滚

每次测试从新数据库开始，先提交“已有”，再请求同时写入“待回滚”和重复的“已有”。检查失败后只剩原有记录，避免只检查 409 而漏掉部分写入。

In [14]:
with TestClient(sync_app) as client:
    assert client.post("/notes", json=["已有"]).status_code == 201
    failed = client.post("/notes", json=["待回滚", "已有"])
    remaining = client.get("/notes").json()
assert failed.status_code == 409
assert remaining == [{"id": 1, "title": "已有"}]
print("同步：", failed.status_code, remaining)  # 预期：同步： 409 [{'id': 1, 'title': '已有'}]。

同步： 409 [{'id': 1, 'title': '已有'}]


异步接口保持同一批输入和同样的通过条件，事务上下文负责执行回滚。

In [15]:
async with LifespanManager(async_app) as manager:
    transport = httpx.ASGITransport(app=manager.app)
    async with httpx.AsyncClient(transport=transport, base_url="http://test") as client:
        async with asyncio.timeout(5):
            assert (await client.post("/notes", json=["已有"])).status_code == 201
            failed = await client.post("/notes", json=["待回滚", "已有"])
            remaining = (await client.get("/notes")).json()
assert failed.status_code == 409
assert remaining == [{"id": 1, "title": "已有"}]
print("异步：", failed.status_code, remaining)  # 预期：异步： 409 [{'id': 1, 'title': '已有'}]。

异步： 409 [{'id': 1, 'title': '已有'}]


若直接调用 commit()，写入失败后要显式 await session.rollback()，才能继续使用这个会话。下面先 flush 一条新记录，让它确实进入数据库事务，再制造冲突，最后用同一会话重新查询。

In [16]:
async with LifespanManager(async_app):
    async with async_app.state.sessions() as session:
        session.add(Note(title="已有"))
        await session.commit()
        session.add(Note(title="待回滚"))
        await session.flush()
        session.add(Note(title="已有"))
        try:
            await session.commit()
        except IntegrityError:
            await session.rollback()
        titles = (await session.scalars(select(Note.title))).all()
        assert titles == ["已有"]
        print("显式回滚后同一会话可继续查询：", titles)  # 预期：显式回滚后同一会话可继续查询： ['已有']。

显式回滚后同一会话可继续查询： ['已有']


## 5 两个并发任务各自创建会话

AsyncSession 保存当前事务与对象状态，不能让多个并发任务共同操作。同一请求启动多个数据库任务时，也应分别调用共享工厂创建各自的会话。

![共享工厂，不共享并发任务的会话](image/illustration/21-01-session-ownership.svg)

图示：依据 SQLAlchemy 2.0 并发会话原则自行绘制。图表示对象责任，不表示每个会话永久绑定一条不同物理连接。

下面每个 query\_in\_task 各查一个标题；后续 TaskGroup 并发执行后，比较两会话身份与查询结果。返回会话引用只用于身份核对，离开 async with 时已关闭它们，不再使用这些引用发起查询。

In [17]:
async def query_in_task(factory, title: str):
    async with factory() as session:
        value = await session.scalar(select(Note.title).where(Note.title == title))
        return session, value

先提交两条固定数据，再用 TaskGroup 启动两个只读任务。退出任务组时等待任务结束；asyncio.timeout 限制整个任务组的等待时间。SQLite 同时只允许一个写入者，这个只读实验不测并发写入容量。

In [18]:
async with LifespanManager(async_app):
    factory = async_app.state.sessions
    async with factory.begin() as session:
        session.add_all([Note(title="任务甲"), Note(title="任务乙")])
    async with asyncio.timeout(5):
        async with asyncio.TaskGroup() as group:
            first = group.create_task(query_in_task(factory, "任务甲"))
            second = group.create_task(query_in_task(factory, "任务乙"))
    first_session, first_title = first.result()
    second_session, second_title = second.result()
    assert first_session is not second_session
    assert [first_title, second_title] == ["任务甲", "任务乙"]
    print("两个不同会话：", first_session is not second_session)  # 预期：两个不同会话： True。
    print([first_title, second_title])  # 预期：['任务甲', '任务乙']。

两个不同会话： True
['任务甲', '任务乙']


## 6 显式加载数据，按归属关闭资源

expire_on_commit=False 保留提交时已有的属性值，不保证它们永远与数据库最新值一致，也不会替未加载的关系自动查询。访问过期属性、延迟列或懒加载关系，可能在普通属性读取处触发隐式 I/O。

需要重新读取时显式 await session.refresh()。关系数据可在查询时通过 selectinload 等方式预先加载，或按 async API 显式加载；本章保持单表，不引入关系模型。

In [19]:
async with LifespanManager(async_app):
    async with async_app.state.sessions() as session:
        note = Note(title="显式加载")
        session.add(note)
        await session.commit()
        assert note.title == "显式加载"
        await session.refresh(note, attribute_names=["title"])
        print({"id": note.id, "title": note.title})  # 预期：{'id': 1, 'title': '显式加载'}。
# 提交后读取已加载值；refresh 是明确的、可 await 的数据库读取。

{'id': 1, 'title': '显式加载'}


会话关闭会归还自己借用的连接；engine.dispose() 关闭池中已归还的连接，不能替仍在使用连接的会话完成关闭。异步引擎需要 await，不能依赖垃圾回收代替异步清理。

本章依次退出请求或任务的会话范围、退出生命周期释放连接池、退出临时目录上下文。两个应用最后使用的目录都应消失。

In [20]:
for application in (sync_app, async_app):
    assert not application.state.database_path.parent.exists()
print("同步与异步应用的临时数据库目录均已清理")  # 预期：同步与异步应用的临时数据库目录均已清理。

同步与异步应用的临时数据库目录均已清理


## 本章小结

（1）AsyncEngine、会话工厂与 AsyncSession 职责不同。引擎和工厂供应用使用，每个请求或并发任务拥有自己的会话。

（2）异步数据库调用需要在对应 API 处 await；add 和缓冲结果的 all 不因此变成异步操作。

（3）事务失败要检查实际数据库内容。begin 上下文自动回滚；直接提交失败后，继续使用同一会话前显式 rollback。

（4）已加载属性、过期属性和懒加载关系要分别考虑。先关闭会话，再 await 引擎清理，最后删除临时文件。

自查：为什么共享工厂可以而共享一个并发会话不可以？close、rollback 和 dispose 分别处理什么状态？

## 练习

### 1 同步与异步对照

给两个接口相同输入“准备”“执行”“完成”，核对 POST 201、GET 的顺序和内容一致，生命周期结束后临时目录均被删除。

In [21]:
# 每个应用从独立空数据库开始；使用本章的 sync_app 与 async_app。
# 核对三条记录的 id、标题顺序和生命周期退出后的目录状态。

### 2 已经 flush 的写入是否保留

先提交标题“已有”。在下一个事务中 flush“新甲”“新乙”，再加入“已有”并提交。捕获 IntegrityError 后显式回滚，使用同一个会话查询剩余标题。解释 flush 与 commit 的区别。

In [22]:
# 使用 async_app 的会话工厂，保持两条新记录属于同一个事务。
# 在查询前执行 rollback；核对本事务的新记录和此前提交的记录各自是否存在。

### 3 三个并发会话

先提交“任务甲”“任务乙”“任务丙”。用 TaskGroup 启动三个 query_in_task，每个查询一个标题。检查会话两两不同、结果与输入对应，所有任务结束后再退出生命周期。

In [23]:
# 每个任务内部调用同一个 factory 创建独立 session。
# 观察三个结果、会话身份和引擎清理顺序。

### 分层提示与解析

第 2 题提示一：flush 发送 SQL，但仍处在当前事务中。提示二：冲突后先 rollback，再查询。解析：最终只有“已有”；“新甲”“新乙”虽已 flush，仍被本次回滚撤销。首次 commit 的“已有”属于此前已完成的事务，不被这次回滚删除。

第 3 题提示：工厂可以共享，会话不能作为三个任务的共同参数。解析：三次工厂调用得到三个不同会话，结果按保存任务引用的顺序对应甲、乙、丙；任务完成的先后顺序不必相同。先等待 TaskGroup 退出，再退出生命周期释放引擎。

## 参考与引用来源

| 网站 | 本章参考内容与定位 |
| --- | --- |
| SQLAlchemy 2.0 官方文档 | [ORM Quick Start](https://docs.sqlalchemy.org/en/20/orm/quickstart.html#declare-models)，DeclarativeBase、Mapped 与 mapped_column；[Unique Constraint](https://docs.sqlalchemy.org/en/20/core/constraints.html#unique-constraint)，列级唯一约束；[Engine Configuration](https://docs.sqlalchemy.org/en/20/core/engines.html#creating-urls-programmatically)，URL.create；[Asyncio](https://docs.sqlalchemy.org/en/20/orm/extensions/asyncio.html) 的 Synopsis - Core、Synopsis - ORM、Using AsyncSession with Concurrent Tasks、Preventing Implicit IO、AsyncSession.scalars 和 AsyncEngine.dispose，异步 API、会话归属、加载与关闭；[Session Basics](https://docs.sqlalchemy.org/en/20/orm/session_basics.html#flushing) 的 Flushing、Framing out a begin / commit / rollback block 与并发安全小节，[Transactions](https://docs.sqlalchemy.org/en/20/orm/session_transaction.html#managing-transactions)，事务上下文和失败后的回滚；[SQLite dialect](https://docs.sqlalchemy.org/en/20/dialects/sqlite.html#aiosqlite) 的 aiosqlite 与 Threading/Pooling Behavior，同步连接条件和异步驱动实现。 |
| aiosqlite 官方文档 | [Details](https://aiosqlite.omnilib.dev/en/stable/#details)，每个连接的后台线程与请求队列。 |
| FastAPI 官方文档 | [Async Tests](https://fastapi.tiangolo.com/advanced/async-tests/)，ASGITransport 与生命周期条件；[Dependencies with yield](https://fastapi.tiangolo.com/tutorial/dependencies/dependencies-with-yield/)，请求依赖的退出；[Concurrency and async / await](https://fastapi.tiangolo.com/async/#very-technical-details)，def 路由的线程池执行；[Testing Events](https://fastapi.tiangolo.com/advanced/testing-events/)，TestClient 上下文中的生命周期。 |
| asgi-lifespan 官方仓库 | [Usage 与 API Reference](https://github.com/florimondmanca/asgi-lifespan#usage)，LifespanManager、manager.app、启动和关闭，以及块内未捕获异常的边界。 |
| Python 3.12 官方文档 | [Task Groups](https://docs.python.org/3.12/library/asyncio-task.html#task-groups) 与 [asyncio.timeout](https://docs.python.org/3.12/library/asyncio-task.html#asyncio.timeout)，等待任务与限制等待；[TemporaryDirectory](https://docs.python.org/3.12/library/tempfile.html#tempfile.TemporaryDirectory)，目录的上下文清理。 |
| SQLite 官方文档 | [Isolation In SQLite](https://www.sqlite.org/isolation.html)，写入串行化与同时仅一个写入者。 |